In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

Task 1

In [2]:
df = pd.read_csv("movies.csv")

df = df[["overview", "tagline", "keywords", "genres", "vote_average"]]
df = df.dropna(subset=["overview", "tagline", "keywords"], how="all")

df.head()

,overview,tagline,keywords,genres,vote_average
0,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,culture clash future space war space colony so...,Action Adventure Fantasy Science Fiction,7.2
1,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",ocean drug abuse exotic island east india trad...,Adventure Fantasy Action,6.9
2,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,spy based on novel secret agent sequel mi6,Action Adventure Crime,6.3
3,Following the death of District Attorney Harve...,The Legend Ends,dc comics crime fighter terrorist secret ident...,Action Crime Drama Thriller,7.6
4,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",based on novel mars medallion space travel pri...,Action Adventure Science Fiction,6.1


In [3]:
import re

for col in ["overview", "tagline", "keywords"]:
    df[col] = df[col].fillna("").str.lower()
    df[col] = df[col].str.replace(r"http\S+|www\S+", "", regex=True)  # remove URLs
    df[col] = df[col].str.replace(r"[^a-z\s]", " ", regex=True)                # remove punctuation & numbers
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()       # remove extra whitespace

In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    shuffle=True,
)

print(len(train_df), len(val_df), len(test_df))

3361 720 721


Task_3

In [5]:
import numpy as np

glove_embeddings = {}
embedding_dim = 200

with open("glove.2024.wikigiga.200d/glove.2024.wikigiga.200d.txt", "r", encoding="utf-8") as f:
    for line in f:
        values = line.strip().split()
        
        # skip invalid lines
        if len(values) != embedding_dim + 1:
            continue
        
        word = values[0]
        
        try:
            vector = np.asarray(values[1:], dtype="float32")
            glove_embeddings[word] = vector
        except:
            continue

print("Loaded embeddings:", len(glove_embeddings))

Loaded embeddings: 1287614


In [6]:
glove_vocab = set(glove_embeddings.keys())
print(len(glove_vocab))

1287614


In [7]:
dataset_vocab = set()

for text in df["overview"]:
    dataset_vocab.update(text.split())

covered = sum(1 for word in dataset_vocab if word in glove_vocab)
coverage = covered / len(dataset_vocab) * 100

print(f"Coverage: {coverage:.2f}%")

Coverage: 98.92%


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    vocabulary=glove_vocab
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["overview"])

/home/tanmay/Work/IT549/IT549_Labs/.venv-1/lib/python3.14/site-packages/sklearn/feature_extraction/text.py:1378: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


In [9]:
def doc_embedding(text, tfidf_vectorizer, feature_names):
    vec = np.zeros(embedding_dim)

    row = tfidf_vectorizer.transform([text])
    indices = row.indices
    weights = row.data

    weight_sum = 0

    for idx, weight in zip(indices, weights):
        word = feature_names[idx]

        if word in glove_embeddings:
            vec += weight * glove_embeddings[word]
            weight_sum += weight

    if weight_sum > 0:
        vec /= weight_sum

    return vec

In [10]:
feature_names = tfidf_vectorizer.get_feature_names_out()

X_train_overview = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in train_df["overview"]
])

X_val_overview = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in val_df["overview"]
])

X_test_overview = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in test_df["overview"]
])

print(X_train_overview.shape, X_val_overview.shape, X_test_overview.shape)

(3361, 200) (720, 200) (721, 200)


In [43]:
X_train_tagaline = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in train_df["tagline"]
])

X_val_tagaline = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)        
    for text in val_df["tagline"]
])

X_test_tagaline = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in test_df["tagline"]
])

print(X_train_tagaline.shape, X_val_tagaline.shape, X_test_tagaline.shape)

(3361, 200) (720, 200) (721, 200)


In [ ]:
X_train_keywords = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in train_df["keywords"]
])

X_val_keywords = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)        
    for text in val_df["keywords"]
])  

X_test_keywords = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in test_df["keywords"]
])

print(X_train_keywords.shape, X_val_keywords.shape, X_test_keywords.shape)

(3361, 200) (720, 200) (721, 200)


Task 3

Overview

In [12]:
X_train = X_train_overview
y_train = train_df["vote_average"].values

X_val = X_val_overview
y_val = val_df["vote_average"].values

X_test = X_test_overview
y_test = test_df["vote_average"].values

In [69]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RatingModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
)

    def forward(self, x):
        return self.net(x)

model = RatingModel(200)

In [74]:
y_train= torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), y_train)
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), y_val)
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), y_test)

from torch.utils.data import DataLoader

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

/tmp/ipykernel_153357/3006959804.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train= torch.tensor(y_train, dtype=torch.float32)
/tmp/ipykernel_153357/3006959804.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(y_val, dtype=torch.float32)
/tmp/ipykernel_153357/3006959804.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.float32)


In [75]:
import torch.nn.functional as F

rating_values = torch.arange(1, 11, dtype=torch.float32)
def rating_loss(logits, targets):
    probs = F.softmax(logits, dim=1)
    pred_rating = (probs * rating_values).sum(dim=1)
    return F.mse_loss(pred_rating, targets)

In [76]:
epochs = 12
model = RatingModel(200)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = rating_loss(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = rating_loss(logits, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 74.2149 | Val Loss: 16.4552
Epoch 2 | Train Loss: 71.5330 | Val Loss: 16.5982
Epoch 3 | Train Loss: 70.1163 | Val Loss: 16.5795
Epoch 4 | Train Loss: 68.3064 | Val Loss: 16.6993
Epoch 5 | Train Loss: 66.8904 | Val Loss: 16.7725
Epoch 6 | Train Loss: 63.7191 | Val Loss: 17.2007
Epoch 7 | Train Loss: 61.7531 | Val Loss: 17.1671
Epoch 8 | Train Loss: 59.4327 | Val Loss: 17.6784
Epoch 9 | Train Loss: 56.8996 | Val Loss: 17.1911
Epoch 10 | Train Loss: 54.2190 | Val Loss: 17.0772
Epoch 11 | Train Loss: 51.7126 | Val Loss: 17.9539
Epoch 12 | Train Loss: 50.6637 | Val Loss: 17.8273


In [77]:
model.eval()

preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = F.softmax(logits, dim=1)
        pred_rating = (probs * rating_values).sum(dim=1)

        preds.extend(pred_rating.numpy())
        targets.extend(y_batch.numpy())

mse = mean_squared_error(targets, preds)
rmse = np.sqrt(mse)

print("Test MSE:", mse)
print("Test RMSE:", rmse)

Test MSE: 1.5808002795953677
Test RMSE: 1.2572988028290522


In [78]:
from sklearn.metrics import mean_squared_error
mean_rating = train_df["vote_average"] .mean()

baseline_preds = np.array([mean_rating] * len(test_df))
baseline_mse = mean_squared_error(test_df["vote_average"], baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)

Baseline MSE: 1.4763835417833158
Baseline RMSE: 1.2150652417805867


Tagline

In [85]:
X_train = X_train_tagaline
X_val = X_val_tagaline
X_test = X_test_tagaline

batch_size = 64

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), y_train)
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), y_val)
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [86]:
epochs = 12
model = RatingModel(200)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = rating_loss(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = rating_loss(logits, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 74.4219 | Val Loss: 16.4166
Epoch 2 | Train Loss: 70.7722 | Val Loss: 16.3730
Epoch 3 | Train Loss: 71.2400 | Val Loss: 16.4125
Epoch 4 | Train Loss: 69.0519 | Val Loss: 16.5209
Epoch 5 | Train Loss: 66.9599 | Val Loss: 16.5910
Epoch 6 | Train Loss: 64.6448 | Val Loss: 16.7304
Epoch 7 | Train Loss: 62.3240 | Val Loss: 16.8756
Epoch 8 | Train Loss: 60.7284 | Val Loss: 16.8946
Epoch 9 | Train Loss: 56.5056 | Val Loss: 17.7884
Epoch 10 | Train Loss: 55.3732 | Val Loss: 17.2184
Epoch 11 | Train Loss: 52.8263 | Val Loss: 17.7633
Epoch 12 | Train Loss: 50.9473 | Val Loss: 17.6804


In [87]:
model.eval()

preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = F.softmax(logits, dim=1)
        pred_rating = (probs * rating_values).sum(dim=1)

        preds.extend(pred_rating.numpy())
        targets.extend(y_batch.numpy())

mse = mean_squared_error(targets, preds)
rmse = np.sqrt(mse)

print("Test MSE:", mse)
print("Test RMSE:", rmse)

Test MSE: 1.5910096284272655
Test RMSE: 1.261352301471427


In [88]:
from sklearn.metrics import mean_squared_error
mean_rating = train_df["vote_average"] .mean()

baseline_preds = np.array([mean_rating] * len(test_df))
baseline_mse = mean_squared_error(test_df["vote_average"], baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)

Baseline MSE: 1.4763835417833158
Baseline RMSE: 1.2150652417805867


Keywords

In [89]:
X_train = X_train_keywords
X_val = X_val_keywords
X_test = X_test_keywords

batch_size = 64
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), y_train)
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), y_val)
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [90]:
epochs = 12
model = RatingModel(200)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = rating_loss(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = rating_loss(logits, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 72.3980 | Val Loss: 15.8543
Epoch 2 | Train Loss: 66.0807 | Val Loss: 15.6828
Epoch 3 | Train Loss: 63.8560 | Val Loss: 15.6202
Epoch 4 | Train Loss: 61.1204 | Val Loss: 15.8509
Epoch 5 | Train Loss: 58.5444 | Val Loss: 16.2157
Epoch 6 | Train Loss: 56.3871 | Val Loss: 16.6407
Epoch 7 | Train Loss: 52.4615 | Val Loss: 16.5146
Epoch 8 | Train Loss: 49.7494 | Val Loss: 16.6405
Epoch 9 | Train Loss: 46.2638 | Val Loss: 16.7236
Epoch 10 | Train Loss: 44.0776 | Val Loss: 17.3506
Epoch 11 | Train Loss: 41.3867 | Val Loss: 16.8873
Epoch 12 | Train Loss: 39.9360 | Val Loss: 17.1515


In [91]:
from sklearn.metrics import mean_squared_error
mean_rating = train_df["vote_average"] .mean()

baseline_preds = np.array([mean_rating] * len(test_df))
baseline_mse = mean_squared_error(test_df["vote_average"], baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)

Baseline MSE: 1.4763835417833158
Baseline RMSE: 1.2150652417805867


Task 4

In [107]:
all_genres = set()

for g in df["genres"]:
    if isinstance(g, str):
        all_genres.update(g.split())

all_genres = sorted(list(all_genres))
genre_to_idx = {g: i for i, g in enumerate(all_genres)}

num_genres = len(all_genres)
print("All genres:", all_genres)
print(num_genres)

All genres: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Fiction', 'Foreign', 'History', 'Horror', 'Movie', 'Music', 'Mystery', 'Romance', 'Science', 'TV', 'Thriller', 'War', 'Western']
22


In [111]:
def encode_genres(genre_str):
    vec = np.zeros(num_genres)
    if isinstance(genre_str, str):
        for g in genre_str.split():
            vec[genre_to_idx[g]] = 1
    return vec

y_train = np.array([encode_genres(g) for g in train_df["genres"]])
y_val = np.array([encode_genres(g) for g in val_df["genres"]])
y_test = np.array([encode_genres(g) for g in test_df["genres"]])

Overview

In [112]:
X_train = torch.tensor(X_train_overview, dtype=torch.float32)
X_val = torch.tensor(X_val_overview, dtype=torch.float32)
X_test = torch.tensor(X_test_overview, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
import torch.nn as nn

class GenreModel(nn.Module):
    def __init__(self, input_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_labels)
        )

    def forward(self, x):
        return self.net(x)


In [115]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [119]:
epochs = 12
model = GenreModel(200, num_genres)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 21.8812 | Val Loss: 3.6584
Epoch 2 | Train Loss: 15.7408 | Val Loss: 3.4423
Epoch 3 | Train Loss: 14.2836 | Val Loss: 3.0897
Epoch 4 | Train Loss: 13.2005 | Val Loss: 2.9141
Epoch 5 | Train Loss: 12.5543 | Val Loss: 2.8067
Epoch 6 | Train Loss: 12.0559 | Val Loss: 2.7324
Epoch 7 | Train Loss: 11.7721 | Val Loss: 2.6962
Epoch 8 | Train Loss: 11.5551 | Val Loss: 2.6527
Epoch 9 | Train Loss: 11.3339 | Val Loss: 2.6592
Epoch 10 | Train Loss: 11.1610 | Val Loss: 2.6395
Epoch 11 | Train Loss: 11.0078 | Val Loss: 2.6226
Epoch 12 | Train Loss: 10.8447 | Val Loss: 2.6066


In [120]:
from sklearn.metrics import f1_score, hamming_loss, jaccard_score
import torch

model.eval()

preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        predictions = (probs > 0.5).int()

        preds.extend(predictions.numpy())
        targets.extend(y_batch.numpy())

preds = np.array(preds)
targets = np.array(targets)

In [128]:
micro_f1_overview = f1_score(targets, preds, average="micro")
macro_f1_overview = f1_score(targets, preds, average="macro")
ham_loss_overview = hamming_loss(targets, preds)
jaccard_overview = jaccard_score(targets, preds, average="samples")

print("Micro F1:", micro_f1_overview)
print("Macro F1:", macro_f1_overview)
print("Hamming Loss:", ham_loss_overview)
print("Jaccard Score:", jaccard_overview)

Micro F1: 0.376043557168784
Macro F1: 0.12713527852041082
Hamming Loss: 0.10837221031395788
Jaccard Score: 0.2810811703322106


Tagline

In [122]:
X_train = torch.tensor(X_train_tagaline, dtype=torch.float32)
X_val = torch.tensor(X_val_tagaline, dtype=torch.float32)
X_test = torch.tensor(X_test_tagaline, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

/tmp/ipykernel_153357/3256313166.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
/tmp/ipykernel_153357/3256313166.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(y_val, dtype=torch.float32)
/tmp/ipykernel_153357/3256313166.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.float32)


In [123]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [124]:
epochs = 12
model = GenreModel(200, num_genres)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 23.2323 | Val Loss: 4.0514
Epoch 2 | Train Loss: 16.4260 | Val Loss: 3.5944
Epoch 3 | Train Loss: 15.3804 | Val Loss: 3.5175
Epoch 4 | Train Loss: 14.8478 | Val Loss: 3.4664
Epoch 5 | Train Loss: 14.5531 | Val Loss: 3.3993
Epoch 6 | Train Loss: 14.2750 | Val Loss: 3.3809
Epoch 7 | Train Loss: 14.0540 | Val Loss: 3.4011
Epoch 8 | Train Loss: 13.9204 | Val Loss: 3.3853
Epoch 9 | Train Loss: 13.7412 | Val Loss: 3.3620
Epoch 10 | Train Loss: 13.6008 | Val Loss: 3.3785
Epoch 11 | Train Loss: 13.5232 | Val Loss: 3.4027
Epoch 12 | Train Loss: 13.3259 | Val Loss: 3.4182


In [ ]:
from sklearn.metrics import f1_score, hamming_loss, jaccard_score
model.eval()

preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        predictions = (probs > 0.5).int()

        preds.extend(predictions.numpy())
        targets.extend(y_batch.numpy())

preds = np.array(preds)
targets = np.array(targets)

micro_f1_tagline = f1_score(targets, preds, average="micro")
macro_f1_tagline = f1_score(targets, preds, average="macro")
ham_loss_tagline = hamming_loss(targets, preds)
jaccard_tagline = jaccard_score(targets, preds, average="samples")

print("Micro F1:", micro_f1_tagline)
print("Macro F1:", macro_f1_tagline)
print("Hamming Loss:", ham_loss_tagline)
print("Jaccard Score:", jaccard_tagline)

Micro F1: 0.376043557168784
Macro F1: 0.12713527852041082
Hamming Loss: 0.10837221031395788
Jaccard Score: 0.2810811703322106


Keywords

In [129]:
X_train = torch.tensor(X_train_keywords, dtype=torch.float32)
X_val = torch.tensor(X_val_keywords, dtype=torch.float32)
X_test = torch.tensor(X_test_keywords, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

/tmp/ipykernel_153357/3701757871.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
/tmp/ipykernel_153357/3701757871.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(y_val, dtype=torch.float32)
/tmp/ipykernel_153357/3701757871.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.float32)


In [130]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [131]:
epochs = 12
model = GenreModel(200, num_genres)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1 | Train Loss: 22.1831 | Val Loss: 3.8271
Epoch 2 | Train Loss: 15.5728 | Val Loss: 3.3427
Epoch 3 | Train Loss: 13.8780 | Val Loss: 3.0958
Epoch 4 | Train Loss: 13.1134 | Val Loss: 2.9828
Epoch 5 | Train Loss: 12.5921 | Val Loss: 2.9490
Epoch 6 | Train Loss: 12.2930 | Val Loss: 2.9155
Epoch 7 | Train Loss: 12.0554 | Val Loss: 2.9314
Epoch 8 | Train Loss: 11.9096 | Val Loss: 2.8986
Epoch 9 | Train Loss: 11.7080 | Val Loss: 2.9280
Epoch 10 | Train Loss: 11.6051 | Val Loss: 2.9033
Epoch 11 | Train Loss: 11.4521 | Val Loss: 2.9034
Epoch 12 | Train Loss: 11.3359 | Val Loss: 2.9162


In [133]:
from sklearn.metrics import f1_score, hamming_loss, jaccard_score
model.eval()

preds = []
targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        predictions = (probs > 0.5).int()

        preds.extend(predictions.numpy())
        targets.extend(y_batch.numpy())

preds = np.array(preds)
targets = np.array(targets)

micro_f1_keywords = f1_score(targets, preds, average="micro")
macro_f1_keywords = f1_score(targets, preds, average="macro")
ham_loss_keywords = hamming_loss(targets, preds)
jaccard_keywords = jaccard_score(targets, preds, average="samples")

print("Micro F1:", micro_f1_keywords)
print("Macro F1:", macro_f1_keywords)
print("Hamming Loss:", ham_loss_keywords)
print("Jaccard Score:", jaccard_keywords)

Micro F1: 0.514890800794176
Macro F1: 0.31427085067873345
Hamming Loss: 0.09242214096583029
Jaccard Score: 0.37571659731853907


/home/tanmay/Work/IT549/IT549_Labs/.venv-1/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Task 5

In [144]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

from collections import defaultdict, Counter

genre_word_counts = defaultdict(Counter)

for _, row in df.iterrows():
    text = row["overview"]
    genres = row["genres"]

    if not isinstance(text, str) or not isinstance(genres, str):
        continue

    # remove stopwords
    words = [
        w for w in text.split()
        if w.lower() not in stop_words
    ]

    for g in genres.split():
        genre_word_counts[g].update(words)

[nltk_data] Downloading package stopwords to /home/tanmay/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [145]:
top_words = {}

for genre, counter in genre_word_counts.items():
    top_words[genre] = counter.most_common(10)

bottom_words = {}


for genre, counter in genre_word_counts.items():
    filtered = [(w, c) for w, c in counter.items() if c >= 3]
    filtered_sorted = sorted(filtered, key=lambda x: x[1])
    bottom_words[genre] = filtered_sorted[:10]

In [146]:
for genre in top_words:
    print(f"\nGenre: {genre}")

    top_df = pd.DataFrame(top_words[genre], columns=["Word", "Frequency"])
    bottom_df = pd.DataFrame(bottom_words[genre], columns=["Word", "Frequency"])

    print("\nTop 10 words")
    print(top_df)

    print("\nLeast frequent words (≥3)")
    print(bottom_df)


Genre: Action

Top 10 words
    Word  Frequency
0  world        184
1    one        175
2   must        166
3    new        161
4    man        151
5    two        137
6   life        135
7  young        123
8   find        109
9    war        106

Least frequent words (≥3)
        Word  Frequency
0   believed          3
1     turner          3
2    message          3
3    spectre          3
4      shape          3
5   shifting          3
6   avengers          3
7  alliances          3
8       sort          3
9    closest          3

Genre: Adventure

Top 10 words
    Word  Frequency
0  world        171
1    new        125
2   must        123
3   find        113
4    one        106
5  young        106
6   life        100
7    two         97
8    man         81
9   save         74

Least frequent words (≥3)
          Word  Frequency
0   dispatched          3
1       unique          3
2       orders          3
3   protecting          3
4     believed          3
5        quite          3

#### Note -

1. The most important words reflect the genre. Eg. Love in romance
2. There are some shared words in similar generes. Eg war in Action and War.


Task 6

In [148]:
all_genres = sorted({
    g
    for row in train_df["genres"].dropna()
    for g in row.split()
})

genre_to_idx = {g: i for i, g in enumerate(all_genres)}
num_genres = len(all_genres)

def encode_genres(series):
    Y = np.zeros((len(series), num_genres))
    
    for i, row in enumerate(series):
        if isinstance(row, str):
            for g in row.split():
                if g in genre_to_idx:
                    Y[i, genre_to_idx[g]] = 1
    return Y

y_train = encode_genres(train_df["genres"])
y_val = encode_genres(val_df["genres"])
y_test = encode_genres(test_df["genres"])

In [149]:
tfidf_words = TfidfVectorizer(
    max_features=20000,
    min_df=3
)

X_train_tfidf = tfidf_words.fit_transform(train_df["keywords"])
X_val_tfidf = tfidf_words.transform(val_df["keywords"])
X_test_tfidf = tfidf_words.transform(test_df["keywords"])

feature_names = tfidf_words.get_feature_names_out()

In [150]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(
    LogisticRegression(max_iter=2000)
)

model.fit(X_train_tfidf, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...max_iter=2000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [153]:
from sklearn.metrics import f1_score, hamming_loss, jaccard_score

y_pred_probs = model.predict_proba(X_test_tfidf)

threshold = 0.5
y_pred = (y_pred_probs >= threshold).astype(int)

micro_f1 = f1_score(y_test, y_pred, average="micro")
macro_f1 = f1_score(y_test, y_pred, average="macro")
hamming = hamming_loss(y_test, y_pred)
jaccard = jaccard_score(y_test, y_pred, average="micro")

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)
print("Hamming Loss:", hamming)
print("Jaccard Score:", jaccard)

Micro F1: 0.38226059654631084
Macro F1: 0.18518749963759973
Hamming Loss: 0.09923086622115748
Jaccard Score: 0.23629306162057254


In [151]:
top_k = 10

for i, genre in enumerate(all_genres):
    coef = model.estimators_[i].coef_[0]

    top_indices = np.argsort(coef)[-top_k:][::-1]
    top_words = feature_names[top_indices]

    print("\nGenre:", genre)
    print("Top indicative words:", top_words)


Genre: Action
Top indicative words: ['assassin' 'martial' 'arts' 'dystopia' 'hero' 'comic' 'helicopter'
 'police' 'spy' 'fight']

Genre: Adventure
Top indicative words: ['magic' 'marvel' 'comic' 'treasure' 'hero' 'the' 'england' 'dinosaur'
 'dragon' 'adventure']

Genre: Animation
Top indicative words: ['animation' 'animal' 'fish' 'magic' 'penguin' 'bunny' 'ice' 'squirrel'
 'village' 'wretch']

Genre: Comedy
Top indicative words: ['comedy' 'friendship' 'spoof' 'holiday' 'high' 'christmas' 'slacker'
 'out' 'duringcreditsstinger' 'age']

Genre: Crime
Top indicative words: ['police' 'detective' 'fbi' 'undercover' 'murder' 'drug' 'crime' 'hitman'
 'psychopath' 'gambling']

Genre: Documentary
Top indicative words: ['music' 'documentary' 'director' 'artist' 'woman' 'pop' 'usa' 'culture'
 'contest' 'sport']

Genre: Drama
Top indicative words: ['biography' 'war' 'rape' 'adultery' 'love' 'drama' 'christian'
 'dysfunctional' 'professor' 'court']

Genre: Family
Top indicative words: ['holiday' 'm

The words are more related to particular genres compared to previous analysis. Eg - War is no longer a part of action.